# Chinese Corpus Review

## tl;dr

现用 Ultra-FineWeb 的固定版本与本次查询的上游 main 一致。现有抽取器读取 content/source，未按 score 进一步筛选。FineWeb2-HQ 中文是首个外部对照候选；没有完成新语料的断点质量抽样或模型实验，不能宣称优于现用数据。

## Context & Methods

2026-10-04，核对作者数据卡、版本元数据、本地清单和抽取器。原始网页证据位于 evidence/，index.json 记录固定 revision 和数据卡 SHA256。

### Key Assumptions

本文“质量”是文本完整性与抽取后切分标签的可接受性；不是知识测验成绩。历史 v7 计数尚未应用新的单字过滤。没有用数据集卡规模推断可保留样本数。

环境：Python 3 标准库。build.py 按顺序执行下列代码并捕获输出；未使用 Jupyter 内核，不需要模型依赖或网络。

In [1]:
from pathlib import Path
import ast, hashlib, json
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'training/train_sharded.py').exists())
review = root/'training/reviews/corpus-source-review-20261004'
manifest_path = root/'training/data/processed/padding-fixed-web-350m-v7-20260925/manifest.json'
manifest = json.loads(manifest_path.read_text())
index = json.loads((review/'evidence/index.json').read_text())
assert len(index) == len({x['repository'] for x in index}) == 10
print('Dataset repositories inspected:', len(index))


Dataset repositories inspected: 10


## Data

检查已保存证据的完整性。门控仓库的原始 README 下载被拒绝，明确记录，不将其描述为已下载语料。

In [2]:
verified = 0
for item in index:
    if 'readme_sha256' in item:
        path = review/'evidence'/(item['repository'].replace('/', '--')+'.md')
        assert hashlib.sha256(path.read_bytes()).hexdigest() == item['readme_sha256']
        verified += 1
    else:
        assert item.get('error') and item.get('gated')
print('Pinned README hashes verified:', verified)
print('Gated README downloads unavailable:', len(index)-verified)


Pinned README hashes verified: 8
Gated README downloads unavailable: 2


## Results

### 当前版本、抽取字段与样本份额

检查代码实际传给 Parquet 的列；并不把“未进一步使用分数”误解成“上游完全没筛选”。

In [3]:
upstream = next(x for x in index if x['repository'] == manifest['web_source']['repository'])
assert manifest['web_source']['revision'] == upstream['revision']
tree = ast.parse((root/'training/prepare_web_data.py').read_text())
columns = [ast.literal_eval(k.value) for n in ast.walk(tree)
           if isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr == 'iter_batches'
           for k in n.keywords if k.arg == 'columns']
assert columns == [['content', 'source']]
counts = manifest['statistics']['domain_samples']  # historical schema, not semantic domains
total = sum(counts.values())
assert total == manifest['statistics']['samples'] == 396266210
assert counts['web'] == 350000000
print('Pinned Ultra-FineWeb revision equals observed upstream:', upstream['revision'])
print('Parquet columns:', columns)
print('Historical training rows:', total)
print('Historical web rows / all rows: %.4f%%' % (100*counts['web']/total))


Pinned Ultra-FineWeb revision equals observed upstream: 02c85641e3d19a854be2e09139c25adaa9518063
Parquet columns: [['content', 'source']]
Historical training rows: 396266210
Historical web rows / all rows: 88.3245%


### 文档证据核验

仅检查候选子集、字段及授权说明确实出现在作者数据卡；不是对样本语义的自动评分。

In [4]:
def card(name):
    return (review/'evidence'/(name.replace('/', '--')+'.md')).read_text()
hq = card('epfml/FineWeb2-HQ')
assert all(s in hq for s in ['cmn_Hani', '54,211,986', 'quality_score', 'top 10%', 'ODC-By'])
edu = card('opencsg/Fineweb-Edu-Chinese-V2.1')
assert all(s in edu for s in ['17,790,513', '70 GB', '获得许可', '46 billion', '46 亿'])
assert 'CASIA-LM/ChineseWebText2.0' in card('openbmb/Ultra-FineWeb')
print('FineWeb2-HQ Chinese subset and quality/provenance fields: confirmed in card')
print('Edu V2.1: commercial-permission clause present; bilingual token totals conflict')
print('Ultra-FineWeb upstream card already links ChineseWebText2.0')
print('No new corpus quality rates or model results measured.')


FineWeb2-HQ Chinese subset and quality/provenance fields: confirmed in card
Edu V2.1: commercial-permission clause present; bilingual token totals conflict
Ultra-FineWeb upstream card already links ChineseWebText2.0
No new corpus quality rates or model results measured.


## Takeaways

保留现用语料作为基线，加入它自身的高分子集，与 FineWeb2-HQ 中文作独立对照；Edu V2.1 高分档需先澄清商用条件。跨库先去重，再用同一抽取器和两侧至少两字规则检查标签，不恢复来源或位置加权。只有来源配额统计，不将来源标签当语义分类。全量训练仍暂停。